# Simple Transformer Using TensorFlow and Keras

## Learning objective
Build a small Transformer encoder step by step and inspect the shape of the data after each stage.

**Example input:** `I love Python`

The notebook demonstrates token IDs, embeddings, multi-head self-attention, residual connections, layer normalization, a feed-forward network, and vocabulary scores.

> **Important:** This is an educational, untrained demonstration. It does not include positional encoding, causal masking, or model training. Therefore, its predicted word is arbitrary and should not be interpreted as a meaningful next-word prediction.


## 1. Import TensorFlow and Keras layers

We use TensorFlow for tensor operations and Keras layers to build the Transformer components.

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers

print("TensorFlow version:", tf.__version__)

## 2. Create a vocabulary

A vocabulary is the collection of tokens that the model knows. Each word is assigned a unique integer ID because neural networks process numbers rather than raw text.

In [ ]:
# A small vocabulary for this classroom example
words = ["I", "love", "Python", "AI", "Machine", "Learning"]

# Assign a unique numerical ID to every word
word_to_id = {word: i for i, word in enumerate(words)}

print("Vocabulary:", word_to_id)

## 3. Tokenization

Tokenization converts the input words into their corresponding IDs.

For this vocabulary:
- `I` becomes `0`
- `love` becomes `1`
- `Python` becomes `2`

The input tensor has shape `(1, 3)`: one sentence containing three tokens.

In [ ]:
# Original input sentence
sentence = ["I", "love", "Python"]

# Convert each word into its vocabulary ID
input_ids = tf.constant([
    [word_to_id[word] for word in sentence]
])

print("Input token IDs:")
print(input_ids.numpy())
print("Input shape:", input_ids.shape)  # (batch_size, sequence_length) = (1, 3)

## 4. Word embedding

An embedding layer maps each token ID to a dense vector. Here, every token is represented by 8 numbers.

Input shape: `(1, 3)`  
Embedding output shape: `(1, 3, 8)`

In [ ]:
embedding_dim = 8

embedding = layers.Embedding(
    input_dim=len(words),       # Number of tokens in the vocabulary: 6
    output_dim=embedding_dim    # Number of values in each token vector: 8
)

# Convert token IDs into embedding vectors
x = embedding(input_ids)

print("Embedding output:")
print(x.numpy())
print("Embedding shape:", x.shape)  # (1, 3, 8)

## 5. Multi-head self-attention

Self-attention lets each token calculate how much attention to give to other tokens.

In self-attention, the same input is supplied as:
- **Query (Q):** what a token is looking for
- **Key (K):** what each token offers for matching
- **Value (V):** the information that can be combined

Because `query`, `key`, and `value` are all `x`, this is self-attention. With two heads and `key_dim=4`, the concatenated attention output has 8 features.

In [ ]:
attention = layers.MultiHeadAttention(
    num_heads=2,  # Use two attention heads
    key_dim=4     # Query/key dimension for each head
)

# Self-attention: Query = Key = Value = the same embedding tensor
attention_output = attention(
    query=x,
    key=x,
    value=x
)

print("Attention output shape:", attention_output.shape)  # (1, 3, 8)

## 6. Residual connection and layer normalization

A residual connection adds the original input to the attention output. This helps preserve information and supports gradient flow during training.

Layer normalization normalizes the features for more stable learning.

In [ ]:
# Add the original embeddings to the attention output
x = layers.Add()([x, attention_output])

# Normalize the combined representation
x = layers.LayerNormalization()(x)

print("After attention + residual + normalization:", x.shape)  # (1, 3, 8)

## 7. Feed-forward network

The feed-forward network is applied independently to each token representation:
1. The first Dense layer expands the feature size from 8 to 16 and applies ReLU.
2. The second Dense layer projects the features back from 16 to 8.

The sequence and batch dimensions remain unchanged.

In [ ]:
ffn = tf.keras.Sequential([
    layers.Dense(16, activation="relu"),
    layers.Dense(embedding_dim)
])

ffn_output = ffn(x)

print("Feed-forward output shape:", ffn_output.shape)  # (1, 3, 8)

## 8. Second residual connection and normalization

Add the feed-forward output to its input, then apply layer normalization. This completes the basic Transformer encoder block.

In [ ]:
x = layers.Add()([x, ffn_output])
x = layers.LayerNormalization()(x)

print("Final Transformer encoder output shape:", x.shape)  # (1, 3, 8)

## 9. Output layer

A Dense layer maps each token representation to one score per vocabulary word.

There are 6 words in our vocabulary, so the output shape is `(1, 3, 6)`:
- 1 sentence in the batch
- 3 input token positions
- 6 vocabulary scores at each position

In [ ]:
output_layer = layers.Dense(len(words))

# Generate vocabulary scores for every input token position
predictions = output_layer(x)

print("Prediction tensor shape:", predictions.shape)  # (1, 3, 6)
print("Scores for all token positions:")
print(predictions.numpy())

## 10. Select a predicted token

For demonstration, select the scores at the final input position (`Python`) and choose the vocabulary ID with the highest score.

**This is not a trained next-word predictor.** The embedding, attention, and Dense layer weights are randomly initialized, so the selected word is arbitrary.

In [ ]:
# Select the output scores for the last input token
last_token_output = predictions[:, -1, :]

# Find the vocabulary ID with the highest score
predicted_id = tf.argmax(
    last_token_output,
    axis=-1
).numpy()[0]

# Convert the predicted ID back to its word
predicted_word = words[predicted_id]

print("Input sentence:", " ".join(sentence))
print("Demonstration output word:", predicted_word)
print("Note: This output is arbitrary because the model has not been trained.")

## 11. Review the tensor shapes

| Stage | Shape | Meaning |
|---|---|---|
| Input IDs | `(1, 3)` | 1 sentence, 3 tokens |
| Embedding | `(1, 3, 8)` | 8 features per token |
| Attention | `(1, 3, 8)` | Context-mixed token representations |
| Feed-forward | `(1, 3, 8)` | Processed token representations |
| Encoder output | `(1, 3, 8)` | Final encoder representations |
| Prediction scores | `(1, 3, 6)` | 6 vocabulary scores per token |

## 12. Practice questions

1. Change `embedding_dim` from 8 to 16. What other layer settings must be updated?
2. Change `num_heads` from 2 to 4. What must be true about the attention output dimension?
3. What are Query, Key, and Value?
4. Why is a residual connection used?
5. What does Layer Normalization do?
6. Why does the feed-forward network project from 8 to 16 and back to 8?
7. Why does a real Transformer need positional information?
8. Why is the predicted word in this notebook not meaningful?

## Key takeaway

A basic Transformer encoder block uses **self-attention**, a **feed-forward network**, **residual connections**, and **layer normalization** to produce contextual token representations.